# Notebook 02
* Author: Melissa Ormeño
* Date: May 2024

Aims:
* Lee las mutaciones somaticas desde el icgc por tejido canceroso
* Datos descargados del icgc junio-2023

In [1]:
# Dependences


# Step 1: Read Kinase Data
Dataframe from 01.Uniprot_get_TKL.ipynb

In [2]:
# Datos con la lista de genes quinasa a trabajar 
import pandas as pd
interpro_code = 'IPR000333'
df_tkl = pd.read_csv('../data/Genes/df_kinasa_{0}.csv'.format(interpro_code))

In [3]:
lista_tkl_genes = list(df_tkl['Gen_names'].unique())
print(len(lista_tkl_genes))
print(df_tkl.shape)
df_tkl.head()

12
(12, 8)


,Gen_names,Uniprot Name,Aliases,Uniprot ID,NCBI ID,ENS ID,Group,Family
0,BMPR1B,BMR1B_HUMAN,"ALK-6, ALK6, AMD3, AMDD, BDA1D, BDA2, CDw293",O00238,658,ENSG00000138696,TKL,STKR
1,ACVR2A,AVR2A_HUMAN,"ACTRII, ACVR2",P27037,92,ENSG00000121989,TKL,STKR
2,BMPR1A,BMR1A_HUMAN,"10q23del, ACVRLK3, ALK-3, ALK3, BMPR-1A, CD292...",P36894,657,ENSG00000107779,TKL,STKR
3,ACVR1B,ACV1B_HUMAN,"ACTRIB, ACVRLK4, ALK4, SKR2",P36896,91,ENSG00000135503,TKL,STKR
4,TGFBR1,TGFR1_HUMAN,"AAT5, ACVRLK4, ALK-5, ALK5, ESS1, LDS1, LDS1A,...",P36897,7046,ENSG00000106799,TKL,STKR


In [5]:
#DF de mutaciones en TGF-B
family_name = 'STKR'
df_tgfb = pd.read_csv('../data/mutaciones/df_count_mut_{0}.csv'.format(family_name))
#lista de pacientes con mutaciones en tgf-b
lista_tgfb = df_tgfb['icgc_donor_id'].unique()
print(df_tgfb['icgc_donor_id'].nunique())
print(df_tgfb.shape)
print(df_tgfb.columns)
df_tgfb

1061
(3321, 7)
Index(['Gen_names', 'icgc_donor_id', 'aa_mutation', 'consequence_type',
       'tipo_cancer', 'total_read_count', 'mutant_allele_read_count'],
      dtype='object')


,Gen_names,icgc_donor_id,aa_mutation,consequence_type,tipo_cancer,total_read_count,mutant_allele_read_count
0,ACVR1,DO10132,C81Y,missense_variant,Colon,33.0,6.0
1,ACVR1,DO10880,D403N,missense_variant,Cerebro,98.0,49.0
2,ACVR1,DO10880,V214A,missense_variant,Cerebro,101.0,52.0
3,ACVR1,DO13522,L337S,missense_variant,Cerebro,67.0,50.0
4,ACVR1,DO14208,W412L,missense_variant,Cabeza,34.0,3.0
...,...,...,...,...,...,...,...
3316,TGFBR2,DO8490,R403H,missense_variant,Colon,180.0,7.0
3317,TGFBR2,DO8490,R428H,missense_variant,Colon,180.0,7.0
3318,TGFBR2,DO8594,R497*,stop_gained,Colon,110.0,15.0
3319,TGFBR2,DO8594,R522*,stop_gained,Colon,110.0,15.0


# Etapa 1: Extraer datos de proteinas TKL, de la data del ICGC
El Consorcio Internacional del Genoma de Cáncer (ICGC) es una organización científica voluntaria que proporciona un foro para colaboración entre los principales investigadores genómicos del mundo del cáncer


In [5]:
len(lista_tgfb)

1061

# 2. Piel

In [40]:
# These row take about 15 minutes
import pandas as pd

# Lista de tipos de cáncer y nombres de archivos correspondientes
cancer_archivos = {
    'Piel': '../data/ICGC/icgc-dataset-skin/donor_therapy.tsv.gz',
    'Higado': '../data/ICGC/icgc-dataset-liver/donor_therapy.tsv.gz',
    'Sangre': '../data/ICGC/icgc-dataset-blood/donor_therapy.tsv.gz',
    'Cerebro': '../data/ICGC/icgc-dataset-brain/donor_therapy.tsv.gz',
    'Seno ': '../data/ICGC/icgc-dataset-breast/donor_therapy.tsv.gz',
    #'Ovario': '../data/ICGC/icgc-dataset-ovario/',
    'Pancreas': '../data/ICGC/icgc-dataset-pancreas/donor_therapy.tsv.gz',
    'Prostata': '../data/ICGC/icgc-dataset-prostate/donor_therapy.tsv.gz',
    #'Utero': '../data/ICGC/icgc-dataset-uterus/',
    #'Estomago': '../data/ICGC/icgc-dataset-stomach/',
    'Pulmon': '../data/ICGC/icgc-dataset-lung/donor_therapy.tsv.gz',
    #'Riñon': '../data/ICGC/icgc-dataset-kidney/',
    'Cabeza': '../data/ICGC/icgc-dataset-head/donor_therapy.tsv.gz',
    #'Vesicula': '../data/ICGC/icgc-dataset-gladd/', #vesicula biliar
    #'Colon': '../data/ICGC/icgc-dataset-colonrectal/',
    #'Cervix': '../data/ICGC/icgc-dataset-cervix/',
    #'Hueso': '../data/ICGC/icgc-dataset-bone/',
    #'Vejiga': '../data/ICGC/icgc-dataset-bladder/'
    
    
    # Agrega más tipos de cáncer con sus respectivos nombres de archivo aquí
}

# DataFrame para almacenar los resultados
df_cancer = pd.DataFrame()

# Itera sobre los tipos de cáncer y archivos correspondientes
for tipo_cancer, archivo in cancer_archivos.items():
    # Lee el archivo
    df_icgc = pd.read_csv(archivo, sep='\t')
    
    # Realiza el merge con df_tkl    #Busca en DF, las mutaciones correspondientes a todos los pacientes con mutaciones en TGF-B
    merge_df = df_icgc.loc[~df_icgc['icgc_donor_id'].isin(lista_tgfb)]

    
    # Agrega una columna con el tipo de cáncer correspondiente
    merge_df['tipo_cancer'] = tipo_cancer
    df_icgc['tipo_cancer'] = tipo_cancer
    
    # Concatena los resultados
    df_cancer = pd.concat([df_cancer, merge_df])

# Muestra el resultado
df_cancer


/var/folders/17/bt4jbx0920sgdvwfqnkx2bnr0000gn/T/ipykernel_41940/1404441962.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merge_df['tipo_cancer'] = tipo_cancer
/var/folders/17/bt4jbx0920sgdvwfqnkx2bnr0000gn/T/ipykernel_41940/1404441962.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  merge_df['tipo_cancer'] = tipo_cancer
/var/folders/17/bt4jbx0920sgdvwfqnkx2bnr0000gn/T/ipykernel_41940/1404441962.py:42: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try

,icgc_donor_id,project_code,submitted_donor_id,first_therapy_type,first_therapy_therapeutic_intent,first_therapy_start_interval,first_therapy_duration,first_therapy_response,second_therapy_type,second_therapy_therapeutic_intent,second_therapy_start_interval,second_therapy_duration,second_therapy_response,other_therapy,other_therapy_response,tipo_cancer
0,DO220846,MELA-AU,MELA-0008,surgery,curative,177.0,1.0,unknown,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Piel
1,DO220845,MELA-AU,MELA-0007,surgery,adjuvant,16.0,1.0,complete response,surgery,curative,1544.0,1.0,unknown,NaN,NaN,Piel
2,DO220848,MELA-AU,MELA-0011,surgery,curative,413.0,1.0,unknown,surgery,curative,499.0,1.0,unknown,NaN,NaN,Piel
3,DO220847,MELA-AU,MELA-0009,surgery,curative,20.0,1.0,unknown,surgery,curative,848.0,1.0,unknown,NaN,NaN,Piel
4,DO220842,MELA-AU,MELA-0002,surgery,curative,228.0,1.0,unknown,surgery,curative,277.0,1.0,unknown,NaN,NaN,Piel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8,DO218012,ORCA-IN,OSCC-GB_011301,radiation therapy,adjuvant,81.0,46.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza
9,DO218180,ORCA-IN,OSCC-GB_012501,radiation therapy,adjuvant,74.0,43.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza
10,DO218280,ORCA-IN,OSCC-GB_011901,radiation therapy,adjuvant,93.0,41.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza
11,DO218211,ORCA-IN,OSCC-GB_012601,radiation therapy,adjuvant,98.0,44.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza


In [41]:
print(df_cancer['icgc_donor_id'].nunique())

1265


In [42]:
print(df_cancer.isnull().sum())

icgc_donor_id                          0
project_code                           0
submitted_donor_id                     0
first_therapy_type                     0
first_therapy_therapeutic_intent       0
first_therapy_start_interval         762
first_therapy_duration               782
first_therapy_response               512
second_therapy_type                    0
second_therapy_therapeutic_intent      0
second_therapy_start_interval        942
second_therapy_duration              975
second_therapy_response              923
other_therapy                        813
other_therapy_response               813
tipo_cancer                            0
dtype: int64


In [43]:
print(df_cancer.shape)
print(df_cancer.columns)

(1266, 16)
Index(['icgc_donor_id', 'project_code', 'submitted_donor_id',
       'first_therapy_type', 'first_therapy_therapeutic_intent',
       'first_therapy_start_interval', 'first_therapy_duration',
       'first_therapy_response', 'second_therapy_type',
       'second_therapy_therapeutic_intent', 'second_therapy_start_interval',
       'second_therapy_duration', 'second_therapy_response', 'other_therapy',
       'other_therapy_response', 'tipo_cancer'],
      dtype='object')


In [44]:
len(df_cancer['tipo_cancer'].unique())

9

In [45]:
df_cancer.shape

(1266, 16)

In [46]:
df_cancer

,icgc_donor_id,project_code,submitted_donor_id,first_therapy_type,first_therapy_therapeutic_intent,first_therapy_start_interval,first_therapy_duration,first_therapy_response,second_therapy_type,second_therapy_therapeutic_intent,second_therapy_start_interval,second_therapy_duration,second_therapy_response,other_therapy,other_therapy_response,tipo_cancer
0,DO220846,MELA-AU,MELA-0008,surgery,curative,177.0,1.0,unknown,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Piel
1,DO220845,MELA-AU,MELA-0007,surgery,adjuvant,16.0,1.0,complete response,surgery,curative,1544.0,1.0,unknown,NaN,NaN,Piel
2,DO220848,MELA-AU,MELA-0011,surgery,curative,413.0,1.0,unknown,surgery,curative,499.0,1.0,unknown,NaN,NaN,Piel
3,DO220847,MELA-AU,MELA-0009,surgery,curative,20.0,1.0,unknown,surgery,curative,848.0,1.0,unknown,NaN,NaN,Piel
4,DO220842,MELA-AU,MELA-0002,surgery,curative,228.0,1.0,unknown,surgery,curative,277.0,1.0,unknown,NaN,NaN,Piel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8,DO218012,ORCA-IN,OSCC-GB_011301,radiation therapy,adjuvant,81.0,46.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza
9,DO218180,ORCA-IN,OSCC-GB_012501,radiation therapy,adjuvant,74.0,43.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza
10,DO218280,ORCA-IN,OSCC-GB_011901,radiation therapy,adjuvant,93.0,41.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza
11,DO218211,ORCA-IN,OSCC-GB_012601,radiation therapy,adjuvant,98.0,44.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza


In [47]:
df_cancer.columns

Index(['icgc_donor_id', 'project_code', 'submitted_donor_id',
       'first_therapy_type', 'first_therapy_therapeutic_intent',
       'first_therapy_start_interval', 'first_therapy_duration',
       'first_therapy_response', 'second_therapy_type',
       'second_therapy_therapeutic_intent', 'second_therapy_start_interval',
       'second_therapy_duration', 'second_therapy_response', 'other_therapy',
       'other_therapy_response', 'tipo_cancer'],
      dtype='object')

In [48]:
print(df_cancer.isnull().sum())

icgc_donor_id                          0
project_code                           0
submitted_donor_id                     0
first_therapy_type                     0
first_therapy_therapeutic_intent       0
first_therapy_start_interval         762
first_therapy_duration               782
first_therapy_response               512
second_therapy_type                    0
second_therapy_therapeutic_intent      0
second_therapy_start_interval        942
second_therapy_duration              975
second_therapy_response              923
other_therapy                        813
other_therapy_response               813
tipo_cancer                            0
dtype: int64


In [49]:
# Eliminar columnas que son completamente NaN
df_cancer_cleaned = df_cancer.dropna(axis=1, how='all')
df_cancer_cleaned

,icgc_donor_id,project_code,submitted_donor_id,first_therapy_type,first_therapy_therapeutic_intent,first_therapy_start_interval,first_therapy_duration,first_therapy_response,second_therapy_type,second_therapy_therapeutic_intent,second_therapy_start_interval,second_therapy_duration,second_therapy_response,other_therapy,other_therapy_response,tipo_cancer
0,DO220846,MELA-AU,MELA-0008,surgery,curative,177.0,1.0,unknown,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Piel
1,DO220845,MELA-AU,MELA-0007,surgery,adjuvant,16.0,1.0,complete response,surgery,curative,1544.0,1.0,unknown,NaN,NaN,Piel
2,DO220848,MELA-AU,MELA-0011,surgery,curative,413.0,1.0,unknown,surgery,curative,499.0,1.0,unknown,NaN,NaN,Piel
3,DO220847,MELA-AU,MELA-0009,surgery,curative,20.0,1.0,unknown,surgery,curative,848.0,1.0,unknown,NaN,NaN,Piel
4,DO220842,MELA-AU,MELA-0002,surgery,curative,228.0,1.0,unknown,surgery,curative,277.0,1.0,unknown,NaN,NaN,Piel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8,DO218012,ORCA-IN,OSCC-GB_011301,radiation therapy,adjuvant,81.0,46.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza
9,DO218180,ORCA-IN,OSCC-GB_012501,radiation therapy,adjuvant,74.0,43.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza
10,DO218280,ORCA-IN,OSCC-GB_011901,radiation therapy,adjuvant,93.0,41.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza
11,DO218211,ORCA-IN,OSCC-GB_012601,radiation therapy,adjuvant,98.0,44.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza


In [50]:
df_cancer

,icgc_donor_id,project_code,submitted_donor_id,first_therapy_type,first_therapy_therapeutic_intent,first_therapy_start_interval,first_therapy_duration,first_therapy_response,second_therapy_type,second_therapy_therapeutic_intent,second_therapy_start_interval,second_therapy_duration,second_therapy_response,other_therapy,other_therapy_response,tipo_cancer
0,DO220846,MELA-AU,MELA-0008,surgery,curative,177.0,1.0,unknown,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Piel
1,DO220845,MELA-AU,MELA-0007,surgery,adjuvant,16.0,1.0,complete response,surgery,curative,1544.0,1.0,unknown,NaN,NaN,Piel
2,DO220848,MELA-AU,MELA-0011,surgery,curative,413.0,1.0,unknown,surgery,curative,499.0,1.0,unknown,NaN,NaN,Piel
3,DO220847,MELA-AU,MELA-0009,surgery,curative,20.0,1.0,unknown,surgery,curative,848.0,1.0,unknown,NaN,NaN,Piel
4,DO220842,MELA-AU,MELA-0002,surgery,curative,228.0,1.0,unknown,surgery,curative,277.0,1.0,unknown,NaN,NaN,Piel
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
8,DO218012,ORCA-IN,OSCC-GB_011301,radiation therapy,adjuvant,81.0,46.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza
9,DO218180,ORCA-IN,OSCC-GB_012501,radiation therapy,adjuvant,74.0,43.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza
10,DO218280,ORCA-IN,OSCC-GB_011901,radiation therapy,adjuvant,93.0,41.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza
11,DO218211,ORCA-IN,OSCC-GB_012601,radiation therapy,adjuvant,98.0,44.0,complete response,no treatment,not applicable,NaN,NaN,NaN,NaN,NaN,Cabeza


In [51]:
#interpro_code = 'IPR000333'
family_name = 'donor_teraphy_no_tgf'
ruta = '../data/Cancer/df_icgc_{0}.csv'.format(family_name)
df_cancer.to_csv(ruta, index=False)